# 均线交叉策略：从数据到可复用回测系统

本 notebook 用一个经典的 MA 双均线交叉策略，演示完整量化研究流程：数据获取、信号生成、交易执行、绩效评估、可视化和参数敏感性分析。

核心假设：短均线上穿长均线说明趋势可能转强，短均线下穿长均线说明趋势可能转弱。为了避免未来函数，本 notebook 默认在当日收盘后确认信号，并在下一交易日开盘执行。

## 1. 导入依赖

项目里的 `quant_lab` 是可复用回测包，notebook 和网站后端会共用它。这样以后新增策略时，只需要新增策略类，而不是重写整套回测流程。

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# 鲁棒地查找包含 quant_lab 包的项目根目录（兼容各种 Jupyter 启动目录）
ROOT = Path.cwd().resolve()

# 策略1: 向上查找（适用于 Jupyter 在项目内部启动）
for _ in range(6):
    if (ROOT / "quant_lab").is_dir():
        break
    ROOT = ROOT.parent
else:
    # 策略2: 向下查找（适用于 Jupyter 在项目父目录启动），限制深度避免全盘扫描
    ROOT = None
    start = Path.cwd().resolve()
    try:
        for dirpath, dirnames, _ in os.walk(start, topdown=True):
            # 限制搜索深度为 5 层
            depth = len(Path(dirpath).relative_to(start).parts)
            if depth > 4:
                dirnames.clear()
                continue
            # 跳过隐藏目录和系统目录，避免权限问题
            dirnames[:] = [d for d in dirnames if not d.startswith(".") and d not in {"Library", "Applications", "System", "node_modules", "__pycache__"}]
            if "quant_lab" in dirnames:
                ROOT = Path(dirpath)  # 项目根目录（quant_lab 的父目录）
                break
    except (PermissionError, OSError):
        pass

    if ROOT is None or not (ROOT / "quant_lab").is_dir():
        raise FileNotFoundError(
            "找不到 quant_lab 包。请确认项目目录结构正确，"
            "或在 Jupyter 中切换到项目根目录后再运行。"
        )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from quant_lab import BacktestConfig, fetch_rqdata_price, load_csv_price, optimize_ma_cross, run_backtest
from quant_lab.plotting import plot_ma_cross_result

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## 2. 设置研究参数

这些参数之后也会映射到网站的 Strategy Lab：标的、时间窗口、复权、资金、手续费、滑点、快慢均线和目标仓位都可以调整。

In [ ]:
SYMBOL = "002050.XSHE"
START_DATE = "2025-07-01"
END_DATE = "2026-07-03"
FREQUENCY = "1d"
ADJUST_TYPE = "pre"

FAST_WINDOW = 5
SLOW_WINDOW = 20
TARGET_WEIGHT = 0.95

INITIAL_CASH = 100_000
COMMISSION_RATE = 0.0008
SLIPPAGE_RATE = 0.0002
MIN_COMMISSION = 5

LOCAL_CSV = ROOT / "data" / "sanhua_002050_xshe_daily_last_year.csv"


## 3. 读取行情数据

优先使用 RQData。如果当前机器没有可用授权或网络不可用，则回退到项目已有 CSV 样本，保证教学流程可以继续跑完。

In [ ]:
try:
    price = fetch_rqdata_price(
        SYMBOL,
        start_date=START_DATE,
        end_date=END_DATE,
        frequency=FREQUENCY,
        adjust_type=ADJUST_TYPE,
    )
    data_source = "RQData"
except Exception as exc:
    print(f"RQData 暂不可用，回退到本地 CSV：{exc}")
    price = load_csv_price(str(LOCAL_CSV), symbol=SYMBOL)
    data_source = "local CSV"

price.head(), price.tail(), data_source


## 4. 数据诊断

正式回测前先检查数据质量。量化研究里，很多错误不是策略逻辑错，而是数据日期、缺失值、复权或重复记录没有处理清楚。

In [ ]:
diagnostics = pd.Series(
    {
        "数据源": data_source,
        "开始日期": price["date"].min().strftime("%Y-%m-%d"),
        "结束日期": price["date"].max().strftime("%Y-%m-%d"),
        "记录数": len(price),
        "重复日期数": int(price["date"].duplicated().sum()),
        "缺失值数量": int(price[["open", "high", "low", "close", "volume"]].isna().sum().sum()),
        "首日收盘": float(price["close"].iloc[0]),
        "最新收盘": float(price["close"].iloc[-1]),
    }
)
diagnostics


## 5. 策略公式

快线均线：

$$
MA_{fast,t} = \frac{1}{N_{fast}}\sum_{i=0}^{N_{fast}-1} Close_{t-i}
$$

慢线均线：

$$
MA_{slow,t} = \frac{1}{N_{slow}}\sum_{i=0}^{N_{slow}-1} Close_{t-i}
$$

当快线从下方向上穿越慢线，生成买入信号；当快线从上方向下穿越慢线，生成卖出信号。

## 6. 运行回测

这里不直接在 notebook 里手写回测循环，而是调用 `quant_lab.run_backtest`。这样 notebook 负责教学展示，真正可复用的逻辑沉淀在项目包里。

In [ ]:
config = BacktestConfig(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    strategy_name="ma_cross",
    strategy_params={
        "fast_window": FAST_WINDOW,
        "slow_window": SLOW_WINDOW,
        "target_weight": TARGET_WEIGHT,
    },
    frequency=FREQUENCY,
    adjust_type=ADJUST_TYPE,
    initial_cash=INITIAL_CASH,
    commission_rate=COMMISSION_RATE,
    slippage_rate=SLIPPAGE_RATE,
    min_commission=MIN_COMMISSION,
    target_weight=TARGET_WEIGHT,
)

result = run_backtest(price, config)
result.notes


## 7. 查看信号表

信号表展示每一天的快线、慢线、信号和目标仓位。注意：信号日不是成交日；成交发生在下一交易日。

In [ ]:
signals = result.signals.copy()
signals[signals["signal"] != 0].tail(10)


## 8. 绩效指标

重点看收益、风险和交易质量，不只看总收益。最大回撤、夏普、卡玛、交易次数和胜率通常比单一收益更能说明策略质量。

In [ ]:
metric_labels = {
    "endingValue": "期末权益",
    "totalReturn": "总收益率",
    "annualizedReturn": "年化收益率",
    "annualizedVolatility": "年化波动率",
    "maxDrawdown": "最大回撤",
    "sharpe": "夏普比率",
    "sortino": "Sortino 比率",
    "calmar": "Calmar 比率",
    "tradeCount": "交易次数",
    "roundTripCount": "完整买卖次数",
    "winRate": "胜率",
    "payoffRatio": "盈亏比",
    "profitFactor": "Profit Factor",
    "averageHoldingBars": "平均持仓天数",
    "commission": "手续费合计",
    "slippage": "滑点成本合计",
    "cost": "总交易成本",
}

metrics = pd.Series(result.metrics).rename(index=metric_labels)
metrics


## 9. 交易流水

交易流水是排查策略最重要的表之一。它能告诉你每笔交易的日期、方向、价格、股数、手续费、滑点和交易后的现金仓位。

In [ ]:
trades = pd.DataFrame([trade.as_dict() for trade in result.trades])
trades.tail(12)


## 10. 可视化

图形分三层：价格和均线、权益曲线、回撤曲线。一个专业回测不能只有收益曲线，必须把回撤放在旁边看。

In [ ]:
fig, axes = plot_ma_cross_result(result.price, result.signals, result.equity)
plt.show()


## 11. 参数敏感性分析

参数优化不是为了找到一个看起来最漂亮的历史参数，而是为了观察策略是否只在极少数组合上有效。如果只有一个参数点好，过拟合风险很高。

In [ ]:
optimization = optimize_ma_cross(
    price,
    config,
    fast_values=range(3, 21, 2),
    slow_values=range(15, 81, 5),
    objective="sharpe",
)
optimization.head(10)


In [ ]:
heatmap = optimization.pivot(index="fast", columns="slow", values="sharpe")
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(heatmap, aspect="auto", cmap="RdYlGn")
ax.set_xticks(range(len(heatmap.columns)))
ax.set_xticklabels(heatmap.columns)
ax.set_yticks(range(len(heatmap.index)))
ax.set_yticklabels(heatmap.index)
ax.set_xlabel("Slow Window")
ax.set_ylabel("Fast Window")
ax.set_title("MA Cross Sharpe Heatmap")
fig.colorbar(image, ax=ax, label="Sharpe")
plt.show()


## 12. 研究小结模板

建议每次回测都写下这几件事：

- 策略逻辑是否有经济含义，而不是只靠历史拟合。
- 信号是否存在未来函数。
- 收益是否主要来自少数几笔交易。
- 最大回撤是否能接受。
- 参数附近是否稳定。
- 手续费和滑点调高后，结果是否仍然成立。

这个 notebook 的结论不构成投资建议，它是学习量化研究流程的可复用模板。